In [1]:
import sqlite3
import os

In [2]:
os.makedirs("data/databases",exist_ok=True)


In [3]:
# creating simple db

con = sqlite3.connect('data/databases/company.db')
cursor = con.cursor()

In [4]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS employee
(id INTEGER PRIMARY KEY, name TEXT, role TEXT, department TEXT, salary REAL )
''')

In [5]:
cursor.execute('''CREATE TABLE IF NOT EXISTS projects
                 (id INTEGER PRIMARY KEY, name TEXT, status TEXT, budget REAL, lead_id INTEGER)''')

In [6]:
# Insert sample data
employees = [
    (1, 'John Doe', 'Senior Developer', 'Engineering', 95000),
    (2, 'Jane Smith', 'Data Scientist', 'Analytics', 105000),
    (3, 'Mike Johnson', 'Product Manager', 'Product', 110000),
    (4, 'Sarah Williams', 'DevOps Engineer', 'Engineering', 98000)
]

projects = [
    (1, 'RAG Implementation', 'Active', 150000, 1),
    (2, 'Data Pipeline', 'Completed', 80000, 2),
    (3, 'Customer Portal', 'Planning', 200000, 3),
    (4, 'ML Platform', 'Active', 250000, 2)
]

In [8]:

cursor.executemany('INSERT OR REPLACE INTO employee VALUES (?,?,?,?,?)', employees)
cursor.executemany('INSERT OR REPLACE INTO projects VALUES (?,?,?,?,?)', projects)

In [9]:
cursor.execute("Select * from employee")

In [10]:
con.commit()
con.close()

In [16]:
# database extraction

from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader


db = SQLDatabase.from_uri("sqlite:///data/databases/company.db")


In [17]:
db.get_usable_table_names()

['employee', 'projects']

In [18]:
db.get_table_info()

'\nCREATE TABLE employee (\n\tid INTEGER, \n\tname TEXT, \n\trole TEXT, \n\tdepartment TEXT, \n\tsalary REAL, \n\tPRIMARY KEY (id)\n)\n\n/*\n3 rows from employee table:\nid\tname\trole\tdepartment\tsalary\n1\tJohn Doe\tSenior Developer\tEngineering\t95000.0\n2\tJane Smith\tData Scientist\tAnalytics\t105000.0\n3\tMike Johnson\tProduct Manager\tProduct\t110000.0\n*/\n\n\nCREATE TABLE projects (\n\tid INTEGER, \n\tname TEXT, \n\tstatus TEXT, \n\tbudget REAL, \n\tlead_id INTEGER, \n\tPRIMARY KEY (id)\n)\n\n/*\n3 rows from projects table:\nid\tname\tstatus\tbudget\tlead_id\n1\tRAG Implementation\tActive\t150000.0\t1\n2\tData Pipeline\tCompleted\t80000.0\t2\n3\tCustomer Portal\tPlanning\t200000.0\t3\n*/'

In [22]:
from typing import List
from langchain_core.documents import Document
# Method 2: Custom SQL to Document conversion
print("\n2️⃣ Custom SQL Processing")

def sql_to_documents(db_path:str)-> List[Document]:
    """Convert SQL Database To documents with context"""
    conn=sqlite3.connect(db_path)
    cursor=conn.cursor()
    documents=[]
    # Strategy 1: Create documents for each table
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table in tables:
        table_name = table[0]
        
        # Get table schema
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        column_names = [col[1] for col in columns]
        
        # Get table data
        cursor.execute(f"SELECT * FROM {table_name}")
        rows = cursor.fetchall()
        
        # Create table overview document
        table_content = f"Table: {table_name}\n"
        table_content += f"Columns: {', '.join(column_names)}\n"
        table_content += f"Total Records: {len(rows)}\n\n"
        
        # Add sample records
        table_content += "Sample Records:\n"
        for row in rows[:5]:  # First 5 records
            record = dict(zip(column_names, row))
            table_content += f"{record}\n"
        
        doc = Document(
            page_content=table_content,
            metadata={
                'source': db_path,
                'table_name': table_name,
                'num_records': len(rows),
                'data_type': 'sql_table'
            }
        )
        documents.append(doc)

     # Strategy 2: Create relationship documents
    # Example: Join employees and projects
    cursor.execute("""
        SELECT e.name, e.role, p.name as project_name, p.status
        FROM employee e
        JOIN projects p ON e.id = p.lead_id
    """)
    
    relationships = cursor.fetchall()
    rel_content = "Employee-Project Relationships:\n\n"
    for rel in relationships:
        rel_content += f"{rel[0]} ({rel[1]}) leads {rel[2]} - Status: {rel[3]}\n"
    
    rel_doc = Document(
        page_content=rel_content,
        metadata={
            'source': db_path,
            'data_type': 'sql_relationships',
            'query': 'employee_project_join'
        }
    )
    documents.append(rel_doc)
    
    conn.close()
    return documents


2️⃣ Custom SQL Processing


In [23]:
sql_to_documents("data/databases/company.db")

[Document(metadata={'source': 'data/databases/company.db', 'table_name': 'employee', 'num_records': 4, 'data_type': 'sql_table'}, page_content="Table: employee\nColumns: id, name, role, department, salary\nTotal Records: 4\n\nSample Records:\n{'id': 1, 'name': 'John Doe', 'role': 'Senior Developer', 'department': 'Engineering', 'salary': 95000.0}\n{'id': 2, 'name': 'Jane Smith', 'role': 'Data Scientist', 'department': 'Analytics', 'salary': 105000.0}\n{'id': 3, 'name': 'Mike Johnson', 'role': 'Product Manager', 'department': 'Product', 'salary': 110000.0}\n{'id': 4, 'name': 'Sarah Williams', 'role': 'DevOps Engineer', 'department': 'Engineering', 'salary': 98000.0}\n"),
 Document(metadata={'source': 'data/databases/company.db', 'table_name': 'projects', 'num_records': 4, 'data_type': 'sql_table'}, page_content="Table: projects\nColumns: id, name, status, budget, lead_id\nTotal Records: 4\n\nSample Records:\n{'id': 1, 'name': 'RAG Implementation', 'status': 'Active', 'budget': 150000.0,